# Dengue en Loreto (2000–2024)
## Notebook 03 · Análisis con SQL

**Entrada:** `data/processed/dengue_loreto_limpio.csv`, generado en `02_limpieza.ipynb`.

**Qué hace este notebook:** carga los casos en una base de datos **SQLite** y responde las preguntas del análisis con consultas SQL: cuándo (años), dónde (provincias y distritos) y qué tan graves son los casos.

**Contenido**
1. Creación de la base de datos
2. Consultas básicas: `SELECT`, `COUNT`, `WHERE`
3. Evolución anual: `GROUP BY`
4. Distribución geográfica
5. Gravedad por año: `SUM`, `AVG` y un cambio de clasificación
6. Comparación de periodos: `CASE WHEN`
7. Años epidémicos: `WHERE` frente a `HAVING`

**Resultado:** `data/dengue_loreto.db` (se genera con este notebook y no se sube a GitHub).

---
## 1. Creación de la base de datos

SQLite guarda toda la base de datos en un solo archivo, sin servidor. `to_sql` copia el DataFrame a una tabla llamada `casos`; con `if_exists="replace"`, el notebook se puede ejecutar varias veces sin errores.

In [1]:
import sqlite3
import pandas as pd

casos = pd.read_csv(
    "../data/processed/dengue_loreto_limpio.csv",
    dtype={"ubigeo": str, "diresa": str}
)

# Conectarse a la base de datos (si el archivo no existe, se crea)
conexion = sqlite3.connect("../data/dengue_loreto.db")

# Copiar el DataFrame a una tabla llamada "casos"
casos.to_sql("casos", conexion, if_exists="replace", index=False)
print("Tabla creada")

Tabla creada


In [2]:
def consulta(sql):
    """Ejecuta una consulta SQL y devuelve el resultado como DataFrame."""
    return pd.read_sql(sql, conexion)

---
## 2. Consultas básicas

`SELECT *` trae todas las columnas y `LIMIT` limita el número de filas: es el equivalente de `df.head()`.

In [3]:
consulta("""
SELECT *
FROM casos
LIMIT 5
""")

,provincia,distrito,ubigeo,ano,semana,diresa,edad_anios,grupo_edad,sexo,gravedad,con_alarma_o_grave
0,ALTO AMAZONAS,YURIMAGUAS,160201,2000,1,16,58.0,Adulto (30-59),M,Sin signos de alarma,0
1,ALTO AMAZONAS,YURIMAGUAS,160201,2000,1,16,44.0,Adulto (30-59),F,Sin signos de alarma,0
2,ALTO AMAZONAS,YURIMAGUAS,160201,2000,1,16,21.0,Joven (18-29),F,Sin signos de alarma,0
3,MAYNAS,IQUITOS,160101,2000,1,16,74.0,Adulto mayor (60+),M,Sin signos de alarma,0
4,MAYNAS,IQUITOS,160101,2000,1,16,42.0,Adulto (30-59),F,Sin signos de alarma,0


`COUNT(*)` cuenta filas. Se usa para verificar que la carga fue completa: deben ser 107,074 casos.

In [4]:
consulta("""
SELECT COUNT(*) AS total_casos
FROM casos
""")

,total_casos
0,107074


`WHERE` filtra filas. En SQL se compara con un solo `=` y el texto va entre comillas simples.

In [5]:
consulta("""
SELECT COUNT(*) AS casos_graves_2024
FROM casos
WHERE ano = 2024
    AND gravedad = 'Grave'
""")

,casos_graves_2024
0,10


---
## 3. Evolución anual

`GROUP BY` agrupa las filas por año y `COUNT(*)` cuenta los casos de cada grupo, como `groupby` en pandas.

In [6]:
consulta("""
SELECT ano, COUNT(*) AS casos
FROM casos
GROUP BY ano
ORDER BY ano
""")

,ano,casos
0,2000,518
1,2001,510
2,2002,2500
3,2003,784
4,2004,2580
5,2005,1772
6,2006,1995
7,2007,1720
8,2008,7232
9,2009,3782


**Hallazgo: Loreto tiene su propia historia epidémica.**

- El año con más casos es **2011 (21,245)**, seguido de 2022 (8,926), 2024 (7,996), 2020 (7,800) y 2008 (7,232).
- A nivel nacional, 2023 y 2024 fueron años récord, pero en Loreto fueron años habituales (**6,582** y **7,996** casos). La ola de 2023–2024 golpeó sobre todo a la costa norte del país.

---
## 4. Distribución geográfica

### 4.1 Casos por provincia

In [7]:
consulta("""
SELECT provincia, COUNT(*) AS casos
FROM casos
GROUP BY provincia
ORDER BY casos DESC;
""")

,provincia,casos
0,MAYNAS,71117
1,ALTO AMAZONAS,22135
2,UCAYALI,3513
3,DATEM DEL MARA�ON,3075
4,REQUENA,2400
5,LORETO,2398
6,MARISCAL RAMON CASTILLA,1706
7,PUTUMAYO,730


> ⚠️ El nombre `DATEM DEL MARA�ON` sigue dañado: la reparación del notebook 01 no encontró una versión limpia de ese nombre. Pendiente: agregarlo al diccionario de correcciones manuales.

### 4.2 Los 10 distritos con más casos

**Regla de SQL:** toda columna del `SELECT` que no esté dentro de una función de agregación (`COUNT`, `SUM`, `AVG`...) debe estar en el `GROUP BY`. SQLite es permisivo con esta regla, pero PostgreSQL, SQL Server y Oracle darían error. Se agrupa por `ubigeo` (el código) y `distrito` (el nombre que se muestra).

In [8]:
consulta("""
SELECT ubigeo, distrito, COUNT(*) AS casos
FROM casos
GROUP BY ubigeo, distrito
ORDER BY casos DESC
LIMIT 10
""")

,ubigeo,distrito,casos
0,160101,IQUITOS,30094
1,160201,YURIMAGUAS,20187
2,160113,SAN JUAN BAUTISTA,16988
3,160108,PUNCHANA,12332
4,160112,BELEN,9790
5,160601,CONTAMANA,2513
6,160701,BARRANCA,2019
7,160301,NAUTA,1561
8,160501,REQUENA,1348
9,160206,LAGUNAS,1269


**Hallazgo: el dengue en Loreto es, sobre todo, un problema urbano de Iquitos.**

- **Maynas concentra el 66.4%** de los casos (71,117) y **Alto Amazonas el 20.7%** (22,135): juntas, casi 9 de cada 10 casos.
- Los cuatro distritos de la ciudad de Iquitos (**Iquitos, San Juan Bautista, Punchana y Belén**) suman **69,204 casos: el 64.6% de Loreto**.
- **Yurimaguas** (20,187 casos) es el segundo distrito con más casos y concentra casi todos los de Alto Amazonas.
- **Precaución:** son conteos absolutos. Para comparar el riesgo entre distritos hace falta la población de cada uno (incidencia).

---
## 5. Gravedad por año

`con_alarma_o_grave` se guardó como 1 (sí) y 0 (no), así que:

- `SUM` cuenta los casos con signos de alarma o graves.
- `AVG` da directamente la **proporción**: el promedio de una columna de unos y ceros es la proporción de unos.

Primero se comprueba que la columna tenga los valores esperados (16,068 con signos de alarma + 554 graves = 16,622).

In [9]:
consulta("""
SELECT con_alarma_o_grave, COUNT(*) AS casos
FROM casos
GROUP BY con_alarma_o_grave
""")

,con_alarma_o_grave,casos
0,0,90452
1,1,16622


In [10]:
consulta("""
SELECT
    ano,
    COUNT(*) AS casos,
    SUM(con_alarma_o_grave) AS casos_alarma_o_grave,
    ROUND(AVG(con_alarma_o_grave) * 100, 1) AS pct_alarma_o_grave
FROM casos
GROUP BY ano
ORDER BY ano
""")

,ano,casos,casos_alarma_o_grave,pct_alarma_o_grave
0,2000,518,0,0.0
1,2001,510,1,0.2
2,2002,2500,1,0.0
3,2003,784,0,0.0
4,2004,2580,5,0.2
5,2005,1772,4,0.2
6,2006,1995,3,0.2
7,2007,1720,18,1.0
8,2008,7232,7,0.1
9,2009,3782,1,0.0


**Hallazgo: un cambio de clasificación en 2010.**

- **Entre 2000 y 2009**, entre el 0.0% y el 1.0% de los casos tiene signos de alarma o es grave. **Desde 2010**, el porcentaje salta a valores de entre 7.7% y 45.9%.
- Un salto tan brusco en un año exacto apunta a un **cambio en cómo se clasifican los casos**, no a un cambio real en la enfermedad. La clasificación actual (*sin signos de alarma, con signos de alarma y grave*) fue propuesta por la OMS en 2009 y adoptada en los años siguientes. Antes se usaban otras categorías, que en este dataset quedaron registradas como "sin signos de alarma".
- **Decisión:** el análisis de gravedad se limita a **2010–2024**.

**Hipótesis: posible sesgo de notificación.** Los años con pocos casos tienen porcentajes altos (2013: 4,479 casos y 45.9%) y el año con más casos tiene uno bajo (2011: 21,245 y 13.4%). En una gran epidemia se registran muchos casos leves, lo que reduce el porcentaje de casos graves; en años tranquilos podrían notificarse sobre todo los casos que llegan a un hospital. Estos datos no permiten confirmarlo.

---
## 6. Comparación de periodos: `CASE WHEN`

`CASE WHEN` crea una categoría nueva dentro de la consulta, como `np.where` en pandas: *"cuando el año es menor que 2010, el periodo es '2000-2009'; en otro caso, '2010-2024'"*.

In [11]:
consulta("""
SELECT
    CASE
        WHEN ano < 2010 THEN '2000-2009'
        ELSE '2010-2024'
    END AS periodo,
    COUNT(*) AS casos,
    SUM(con_alarma_o_grave) AS casos_alarma_o_grave,
    ROUND(AVG(con_alarma_o_grave) * 100, 1) AS pct_alarma_o_grave
FROM casos
GROUP BY periodo
""")

,periodo,casos,casos_alarma_o_grave,pct_alarma_o_grave
0,2000-2009,23393,40,0.2
1,2010-2024,83681,16582,19.8


**Hallazgo:** desde 2010, **el 19.8% de los casos de dengue en Loreto presenta signos de alarma o es grave** (16,582 de 83,681), frente al 0.2% registrado entre 2000 y 2009 con la clasificación anterior.

---
## 7. Años epidémicos: `WHERE` frente a `HAVING`

Los dos filtran, pero en momentos distintos:

- **`WHERE`** filtra **filas**, antes de agrupar.
- **`HAVING`** filtra **grupos**, después de agrupar.

Orden de ejecución de una consulta: `FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT`. Por eso no se puede escribir `WHERE COUNT(*) > 7000`: cuando se ejecuta el `WHERE`, todavía no se ha contado nada.

In [15]:
# Años con más de 7,000 casos
consulta("""
SELECT ano, COUNT(*) AS casos
FROM casos
GROUP BY ano
HAVING COUNT(*) > 7000
ORDER BY ano
""")

,ano,casos
0,2008,7232
1,2011,21245
2,2014,7049
3,2020,7800
4,2022,8926
5,2024,7996


In [19]:
consulta("""
SELECT
    ano,
    COUNT(*) AS casos,
    ROUND(AVG(con_alarma_o_grave) * 100, 1) AS pct_alarma_o_grave
FROM casos
WHERE ano >= 2010
GROUP BY ano
HAVING COUNT(*) > 5000
ORDER BY ano
""")

,ano,casos,pct_alarma_o_grave
0,2011,21245,13.4
1,2014,7049,35.4
2,2020,7800,22.4
3,2021,5115,9.6
4,2022,8926,16.0
5,2023,6582,14.3
6,2024,7996,12.7


In [20]:
consulta("""
SELECT semana, COUNT(*) AS casos
FROM casos
WHERE ano = 2011
GROUP BY semana
ORDER BY semana
""")

,semana,casos
0,1,1305
1,2,2108
2,3,3079
3,4,2518
4,5,1973
5,6,2661
6,7,1295
7,8,1456
8,9,950
9,10,789


In [21]:
consulta("""
WITH semanal AS (
    SELECT ano, semana, COUNT(*) AS casos
    FROM casos
    GROUP BY ano, semana
)
SELECT
    ano,
    semana,
    casos,
    ROUND(casos * 100.0 / SUM(casos) OVER (PARTITION BY ano), 2) AS pct_del_anio
FROM semanal
ORDER BY ano, semana
""")

,ano,semana,casos,pct_del_anio
0,2000,1,10,1.93
1,2000,2,23,4.44
2,2000,3,23,4.44
3,2000,4,21,4.05
4,2000,5,57,11.00
...,...,...,...,...
1298,2024,48,427,5.34
1299,2024,49,430,5.38
1300,2024,50,354,4.43
1301,2024,51,318,3.98


In [22]:
consulta("""
WITH semanal AS (
    SELECT ano, semana, COUNT(*) AS casos
    FROM casos
    GROUP BY ano, semana
),
proporciones AS (
    SELECT
        ano,
        semana,
        casos * 100.0 / SUM(casos) OVER (PARTITION BY ano) AS pct
    FROM semanal
)
SELECT
    semana,
    ROUND(SUM(pct) / (SELECT COUNT(DISTINCT ano) FROM casos), 2) AS pct_promedio
FROM proporciones
GROUP BY semana
ORDER BY semana
""")

,semana,pct_promedio
0,1,3.11
1,2,3.47
2,3,3.37
3,4,3.51
4,5,3.60
5,6,3.58
6,7,3.12
7,8,2.89
8,9,2.52
9,10,2.70
